# SETUP CONFIG DAN S3 BUCKET

In [7]:
import boto3
import sagemaker

# initialize session dan automatically ambil assigned role and default bucket
sm_session = sagemaker.Session()
role = sagemaker.get_execution_role()
bucket = sm_session.default_bucket()
region = boto3.Session().region_name

print(f"Role = {role}")
print(f"Bucket Default = {bucket}")
print(f"Region = {region}")

Role = arn:aws:iam::378178050988:role/LabRole
Bucket Default = sagemaker-us-east-1-378178050988
Region = us-east-1


# PACKAGE ARTIFACT KITA (yang ada di folder models/)

In [8]:
import tarfile
import os

# 3 artifact dari OOP dipanggil
model_files = [
    "models/best_model.pkl",
    "models/preprocessor.pkl",
    "models/model_metadata.pkl"
]

# Compress them into a single archive
with tarfile.open("model.tar.gz", "w:gz") as tar:
    for file_path in model_files:
        if os.path.exists(file_path):
            file_name = os.path.basename(file_path)
            tar.add(file_path, arcname=file_name)
            print(f"Added {file_name} to tarball.")
        else:
            print(f"[ERROR] Cannot find {file_path}. RUN PIPELINE.PY DENGAN BENAR!")

print("model.tar.gz TELAH TERBUAT")

Added best_model.pkl to tarball.
Added preprocessor.pkl to tarball.
Added model_metadata.pkl to tarball.
model.tar.gz TELAH TERBUAT


In [9]:
# UPLOAD KE S3

s3 = boto3.client("s3")

# path in your S3 bucket
s3_key = "credit-score-project/model/model.tar.gz"

print("Uploading to S3...")
s3.upload_file("model.tar.gz", bucket, s3_key)
model_s3_uri = f"s3://{bucket}/{s3_key}"

print(f"model.tar.gz telah terupload : {model_s3_uri}")

Uploading to S3...
model.tar.gz telah terupload : s3://sagemaker-us-east-1-378178050988/credit-score-project/model/model.tar.gz


# === DEPLOY ENDPOINT (BACKEND) ==================

In [10]:
from sagemaker.sklearn.model import SKLearnModel
import json

ENDPOINT_NAME = "daniel-rf-endpoint" 
FRAMEWORK_VERSION = "1.2-1" # VERSI SCIKIT LEARN PASTIKAN SAMA

print("Configuring SKLearnModel container...")
model = SKLearnModel(
    model_data=model_s3_uri,
    role=role,
    entry_point="inference.py", # POINTER KE INFERENCE.PY (inferencing.py lokal versi untuk AWS SageMaker Notebook)
    framework_version=FRAMEWORK_VERSION,
    py_version="py3",
    sagemaker_session=sm_session
)

print("\nDeploying endpoint (This usually takes 5-8 minutes. Grab a coffee!)...")
predictor = model.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    endpoint_name=ENDPOINT_NAME
)
print(f"\n==Deployment Complete!==")





# SMOKE TEST
print("\nInitiating Smoke Test...")
sample_payload = {
    "Age": "30",
    "Occupation": "Scientist",
    "Annual_Income": "50000.0",
    "Num_of_Loan": "2",
    "Outstanding_Debt": "1000.0",
    "Changed_Credit_Limit": "5.0",
    "Credit_History_Age": "10 Years and 0 Months", # 120 months translated
    "Amount_invested_monthly": "150.0",
    "Num_of_Delayed_Payment": "1",
    "Credit_Mix": "Good",
    "Payment_Behaviour": "High_spent_Small_value_payments"
}

sagemaker_payload = {"instances": [sample_payload]}

# memastikan ini match region kita
runtime = boto3.client("sagemaker-runtime", region_name=region) 

try:
    response = runtime.invoke_endpoint(
        EndpointName=ENDPOINT_NAME,
        ContentType="application/json",
        Body=json.dumps(sagemaker_payload),
    )

    print("\nResponse from SageMaker Endpoint:")
    print(response["Body"].read().decode("utf-8"))
    
except Exception as e:
    print(f"\nError invoking endpoint: {e}")

Configuring SKLearnModel container...

Deploying endpoint (This usually takes 5-8 minutes. Grab a coffee!)...
-------!
==Deployment Complete!==

Initiating Smoke Test...

Response from SageMaker Endpoint:
{"predictions": ["Standard"]}


# MATIKAN

In [20]:
sm_client = boto3.client("sagemaker", region_name=region)

print(f"Attempting to delete endpoint: {ENDPOINT_NAME}...")
try:
    sm_client.delete_endpoint(EndpointName=ENDPOINT_NAME)
    print("Endpoint deleted.")
except Exception as e:
    print(f"X {e}")

print(f"Attempting to delete config: {ENDPOINT_NAME}...")
try:
    sm_client.delete_endpoint_config(EndpointConfigName=ENDPOINT_NAME)
    print("Config deleted.")
except Exception as e:
    print(f"X {e}")

Attempting to delete endpoint: daniel-rf-endpoint...
X An error occurred (ValidationException) when calling the DeleteEndpoint operation: Could not find endpoint "daniel-rf-endpoint".
Attempting to delete config: daniel-rf-endpoint...
X An error occurred (ValidationException) when calling the DeleteEndpointConfig operation: Could not find endpoint configuration "daniel-rf-endpoint".


# ========= ISI USERDATA.SH ============

#!/bin/bash
# update OS and install Git and Python
yum update -y
yum install git -y
yum install python3-pip -y

# install SEMUA LIBRARY YANG DIBUTUHKAN
pip3 install streamlit pandas boto3

# CREATE DIREKTORI kita
mkdir -p /home/ec2-user/streamlit_app
cd /home/ec2-user/streamlit_app

# buat baseline streamlit app kita
cat << 'EOF' > app.py
import streamlit as st
import boto3
import json

st.title("Credit Score Predictor - AWS Architecture")
st.write("Frontend: **EC2** | Backend: **SageMaker Endpoint**")

# coba2 dari smoke test kita
sample_payload = {
    "Age": "25",
    "Annual_Income": "50000",
    "Outstanding_Debt": "1500.50",
    "Credit_Mix": "Good",
    "Payment_Behaviour": "High_spent_Medium_value_payments",
    "Credit_History_Age": "5 Years and 2 Months"
}

if st.button("Predict Credit Score"):
    try:
        # Connect to SageMaker via boto3
        runtime = boto3.client("sagemaker-runtime", region_name="us-east-1")
        
        # Send the payload to your specific endpoint
        response = runtime.invoke_endpoint(
            EndpointName="daniel-rf-endpoint",
            ContentType="application/json",
            Body=json.dumps({"instances": [sample_payload]})
        )
        
        result = json.loads(response["Body"].read().decode("utf-8"))
        prediction = result["predictions"][0]
        
        st.success(f"SageMaker Prediction: **{prediction}**")
    except Exception as e:
        st.error(f"Connection Error: {e}")
EOF

# permissions so ec2-user owns the folder
chown -R ec2-user:ec2-user /home/ec2-user/streamlit_app

# STREAMLIT DI RUN DI PORT 8501
su - ec2-user -c "cd /home/ec2-user/streamlit_app && nohup streamlit run app.py --server.port 8501 > streamlit.log 2>&1 &"

# ===== SUPAYA EC2 JALAN (jalanin di terminal EC2) ========

nohup /home/ec2-user/streamlit_app/venv/bin/streamlit run /home/ec2-user/streamlit_app/app.py --server.port 8501 > /home/ec2-user/streamlit_app/streamlit.log 2>&1 &